<a href="https://colab.research.google.com/github/KrathK9722/Analise-de-Dados-Python-Projeto-Avaliativo-M1W07/blob/main/Analise_Google_COLAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1 - Importação das bibliotecas:**

In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

---
## 2 - **Carregamento dos dados**:

In [136]:
id_arquivo = "1iNVCXfu2iFVVHlffBz-fky7U28NhhNv9"

url = f"https://drive.google.com/uc?export=download&id={id_arquivo}"

dados_originais = pd.read_csv(url,sep=";", encoding="latin1")

Sepação feita por ";" porque o padrão estava como "," o que fazia com que o CSV fosse importado com somente uma coluna.

# **3 - Visualização dos dados brutos:**

In [137]:
dados_originais.head(5)

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


# **3.1 - Quantide de Colunas e linhas**

In [138]:
dados_originais.shape

(830000, 14)

-----

### O QUE É CADA COLUNA?

1. DATA: Data da compra;
2. CO_ID: Identificação do número de compra (número da nota fiscal);
3. CL_ID: Identificação do cliente (número do cliente);
4. CL_GENERO: Sexo biológico informado pelo cliente;
5. CL_EC: Estado civil do cliente:
    
    1: Casado ou união estával;
    
    2: Divorciado;
    
    3: Separado;
    
    4: Solteiro;
    
    5: Viúvo.
6. CL_FHL: Número de filhos do cliente;
7. CL_SEG: Segmentação econômica do cliente (classe A, B ou C);
8. PR_ID: Código do produto (SKU) adquirido;
9. PR_CAT: Categoria do produto adquirido;
10. PR_NOME: Nome do produto adquirido.


Demais colunas não contem dados e devem ser removidas no processo de limpeza.

***`Informações retiradas do documento de analise exploratoria da base de varejo.csv`***

-----

# **3.2 - Tipos de dados e outras informações:**

In [139]:
dados_originais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  object 
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  object 
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  object 
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  object 
 9   PR_NOME      830000 non-null  object 
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), object(5)
memory usage: 88.7+ MB


#**4 - Limpeza e Validação dos dados**

Copia do banco de dados original para garantir que os dados originais não sejam modificados e possam ser acessados assim como vieram, permitindo que a tabela seja modificada na nova copia.

In [140]:
dados = pd.read_csv(url,sep=";", encoding="latin1", na_values=["#N/D"])

Modificação na nova tabela para que o pandas entenda que #N/D é nulo e some ao procurar pelos valores nulos.

### **4.1 - Linhas Duplicadas**

In [141]:
duplicados = dados.duplicated()
print(f"Linhas duplicadas: {duplicados.sum()}")

Linhas duplicadas: 96553


Diversos dados duplicados foram encontrados e devem ser removidos para uma analise limpa da tabela.

In [142]:
dados = dados.drop_duplicates()

In [143]:
duplicados_limpos = dados.duplicated()
print(f"Linhas duplicadas: {duplicados_limpos.sum()}")

Linhas duplicadas: 0


Feita a limpeza e remoção dos dados duplicados

---



### **4.2 - Valores Nulos**

In [144]:
nulos = dados.isna().sum()
print(f"Valores nulos: \n{nulos}")

Valores nulos: 
DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT           3228
PR_NOME          3228
Unnamed: 10    733447
Unnamed: 11    733447
Unnamed: 12    733447
Unnamed: 13    733447
dtype: int64


Criação de um dicionário para tentar preencher as categorias e nomes nulos com o valor correto de acordo com o ID do produto procurnado produtos de mesmo ID na tabela para coletar os nomes e categorias referentes ao ID.

In [145]:
mapa_categoria = dados.dropna(subset=["PR_CAT"]).drop_duplicates("PR_ID").set_index("PR_ID")["PR_CAT"]

dados["PR_CAT"] = dados["PR_CAT"].fillna(dados["PR_ID"].map(mapa_categoria))

In [146]:
# Pega quantos produtos unicos tem categoria única
ids_com_nulo = dados.loc[dados["PR_CAT"].isna(), "PR_ID"].unique()
print(f"Quantidade de produtos únicos com categoria nula: {len(ids_com_nulo)}")

# Desses produtos procura quantos tem essa categoria preenchida em outra linha
ids_recuperaveis = dados.loc[dados["PR_ID"].isin(ids_com_nulo) & dados["PR_CAT"].notna(), "PR_ID"].unique()
print(f"Desses, quantos têm categoria em outra linha: {len(ids_recuperaveis)}")

Quantidade de produtos únicos com categoria nula: 1
Desses, quantos têm categoria em outra linha: 0


In [147]:
# Pega quantos produtos únicos têm nome nulo
ids_com_nulo_nome = dados.loc[dados["PR_NOME"].isna(), "PR_ID"].unique()
print(f"Quantidade de produtos únicos com nome nulo: {len(ids_com_nulo_nome)}")

# Desses produtos, quantos têm o nome preenchido em outra linha
ids_recuperaveis_nome = dados.loc[dados["PR_ID"].isin(ids_com_nulo_nome) & dados["PR_NOME"].notna(), "PR_ID"].unique()
print(f"Desses, quantos têm nome em outra linha: {len(ids_recuperaveis_nome)}")

Quantidade de produtos únicos com nome nulo: 1
Desses, quantos têm nome em outra linha: 0


Ao fazer essa visualização dos IDS unicos que tem categoria e nomes nulos podemos entender que todas essas categorias nulas vem de um mesmo produto então precisamos descobrir qual é esse produto.

In [148]:
pr_id_desconhecido = dados.loc[dados["PR_CAT"].isna(), "PR_ID"].unique()[0]
print(f"PR_ID: {pr_id_desconhecido}")

# Confirma que o nome do produto também está nulo
print(dados.loc[dados["PR_ID"] == pr_id_desconhecido, ["PR_ID", "PR_CAT", "PR_NOME"]].head())

PR_ID: 107
     PR_ID PR_CAT PR_NOME
82     107    NaN     NaN
223    107    NaN     NaN
640    107    NaN     NaN
857    107    NaN     NaN
917    107    NaN     NaN


Agora que descobrimos o produto entendemos que esse produto em especifíco esta com algum problema em seus registros que faz com que o seu nome e categoria não estejam informados. Por isso para facilitar visualizações futuras vamos trocar a dados nesses espaços por "Não Informado" para que qualquer um consiga entender ao visualizar os dados.

In [149]:
dados["PR_CAT"] = dados["PR_CAT"].fillna("Não informado")
dados["PR_NOME"] = dados["PR_NOME"].fillna("Não informado")

Após isso vamos agora remover as colunas sem nome para garantir que só o essencial fique na tabela.

In [150]:
dados = dados.drop(columns=["Unnamed: 10", "Unnamed: 11", "Unnamed: 12", "Unnamed: 13"])

In [151]:
nulos = dados.isna().sum()
print(f"Valores nulos: \n{nulos}")

Valores nulos: 
DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64


In [167]:
dados.head(4)

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,DATA_convertida
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,2019-02-01
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,2019-02-01
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,2019-02-01
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,2019-02-01


Após a limpeza e validação de valores nulos e duplicados precisamos validas e converter as Datas de Registro.

### **4.3 - Conversão e Validação das Datas**

In [164]:
dados["DATA_convertida"] = pd.to_datetime(dados["DATA"], format="%d/%m/%Y", errors="coerce")

In [166]:
dados.head(3)

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,DATA_convertida
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,2019-02-01
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,2019-02-01
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,2019-02-01


In [165]:
datas_invalidas = dados["DATA_convertida"].isna().sum()
print(f"Datas inválidas: {datas_invalidas}")

Datas inválidas: 0


Após a conversão e verificação das datas podemos seguir para a próxima etapa que é o agrupamento de informações relevantes e separação correta dos dados para análise.